# SK Producer Key Map Builder (xxhash64)

This notebook builds a key_map between a source SK producer table and a target SK producer table using normalized natural keys and `xxhash64`.

Inputs are provided via widgets for:
- Source catalog, schema, table
- Target catalog, schema, table
- Natural key column(s) (comma-separated for compound keys)
- SCD2 flag

For matching, the notebook uses one hash key per row:
- SCD1 / non-versioned tables: normalized natural keys
- SCD2 tables: normalized natural keys + effective start date

The `natural_key` output is a deterministic concatenation of normalized NK values using `||`.

Optional widgets are provided for explicit SK column names and SCD2 date columns.


In [ ]:
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql import functions as F


In [ ]:
# Databricks widgets
dbutils.widgets.text("source_catalog", "", "Source catalog")
dbutils.widgets.text("source_schema", "", "Source schema")
dbutils.widgets.text("source_table", "", "Source table")

dbutils.widgets.text("target_catalog", "", "Target catalog")
dbutils.widgets.text("target_schema", "", "Target schema")
dbutils.widgets.text("target_table", "", "Target table")

dbutils.widgets.text("natural_keys", "", "Natural key columns (comma-separated)")
dbutils.widgets.dropdown("is_scd2", "false", ["false", "true"], "SCD2 table")

dbutils.widgets.text("source_sk_col", "", "Source SK column (optional)")
dbutils.widgets.text("target_sk_col", "", "Target SK column (optional)")

dbutils.widgets.text("scd2_start_col", "effectiveStartDate", "SCD2 start column")
dbutils.widgets.text("scd2_end_col", "effectiveEndDate", "SCD2 end column")

dbutils.widgets.text("temp_storage_path", "dbfs:/tmp/keymap_builder", "Temp storage path (DBFS or Volume)")

dbutils.widgets.dropdown("save_keymap", "false", ["false", "true"], "Save key_map table")
dbutils.widgets.text("keymap_catalog", "", "Key_map catalog (optional)")

dbutils.widgets.text("keymap_schema", "keymap", "Key_map schema")dbutils.widgets.text("keymap_table", "", "Key_map table (optional)")

In [ ]:
def q(name: str) -> str:
    return dbutils.widgets.get(name).strip()

source_catalog = q("source_catalog")
source_schema = q("source_schema")
source_table = q("source_table")
target_catalog = q("target_catalog")
target_schema = q("target_schema")
target_table = q("target_table")

natural_keys = [c.strip() for c in q("natural_keys").split(",") if c.strip()]
is_scd2 = q("is_scd2").lower() == "true"

source_sk_col_override = q("source_sk_col")
target_sk_col_override = q("target_sk_col")

scd2_start_col = q("scd2_start_col")
scd2_end_col = q("scd2_end_col")

temp_storage_path = q("temp_storage_path").rstrip("/")

save_keymap = q("save_keymap").lower() == "true"
keymap_catalog = q("keymap_catalog")
keymap_schema = q("keymap_schema") or "keymap"
keymap_table = q("keymap_table")

required = {
    "source_catalog": source_catalog,
    "source_schema": source_schema,
    "source_table": source_table,
    "target_catalog": target_catalog,
    "target_schema": target_schema,
    "target_table": target_table,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(f"Missing required widget values: {', '.join(missing)}")
if not temp_storage_path:
    raise ValueError("temp_storage_path must be provided (e.g. dbfs:/tmp/keymap_builder or /Volumes/cat/schema/vol/tmp)")

if save_keymap and not keymap_table:
    raise ValueError("When save_keymap=true, keymap_table must be provided")

source_fqn = f"{source_catalog}.{source_schema}.{source_table}"
target_fqn = f"{target_catalog}.{target_schema}.{target_table}"

print("Source:", source_fqn)
print("Target:", target_fqn)

print("Natural keys:", natural_keys)
print("Natural keys:", natural_keys)print("SCD2:", is_scd2)
print("SCD2:", is_scd2)

In [ ]:
source_df = spark.table(source_fqn)
target_df = spark.table(target_fqn)

def ensure_columns(df: DataFrame, cols: List[str], df_name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing required columns: {missing}")

def detect_sk_col(df: DataFrame, nk_cols: List[str], override: str, side: str) -> str:
    if override:
        if override not in df.columns:
            raise ValueError(f"{side} SK override column '{override}' not found")
        return override

    numeric_types = {"tinyint", "smallint", "int", "bigint", "long"}
    candidates = [
        f.name
        for f in df.schema.fields
        if f.name.lower().startswith("key")
        and f.name not in nk_cols
        and f.dataType.simpleString().lower() in numeric_types
    ]

    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) == 0:
        raise ValueError(
            f"Could not auto-detect {side} SK column. Provide {side.lower()}_sk_col widget."
        )
    raise ValueError(
        f"Multiple possible {side} SK columns found: {candidates}. Provide {side.lower()}_sk_col widget."
    )

def normalize_col(col_name: str) -> F.Column:
    # Normalize NK text deterministically before hashing.
    return F.upper(F.trim(F.coalesce(F.col(col_name).cast("string"), F.lit(""))))

def normalized_key_exprs(cols: List[str]) -> List[F.Column]:
    return [normalize_col(c) for c in cols]

def build_normalized_key_string(cols: List[str]) -> F.Column:
    return F.concat_ws("||", *normalized_key_exprs(cols))

def compute_hashes(df: DataFrame, nk_cols: List[str], scd2: bool) -> DataFrame:
    out = df.withColumn("natural_key", build_normalized_key_string(nk_cols))

    hash_inputs = normalized_key_exprs(nk_cols)
    if scd2:
        hash_inputs = hash_inputs + [normalize_col(scd2_start_col)]

    out = out.withColumn("match_hash", F.xxhash64(*hash_inputs))
    return out

def add_duplicate_metadata(df: DataFrame, side: str, sk_col: str) -> DataFrame:
    grain_cols = ["match_hash"]
    counts = (
        df.groupBy(*grain_cols)
        .agg(F.count(F.lit(1)).alias(f"{side}_row_count"))
    )
    ordered = Window.partitionBy(*grain_cols).orderBy(F.col(sk_col).asc_nulls_last(), F.col("natural_key").asc())
    return (
        df.join(counts, grain_cols, "left")
          .withColumn(f"{side}_row_num", F.row_number().over(ordered))
    )

required_cols = natural_keys.copy()
if is_scd2:
    required_cols += [scd2_start_col, scd2_end_col]

ensure_columns(source_df, required_cols, "source_df")
ensure_columns(target_df, required_cols, "target_df")

source_sk_col = detect_sk_col(source_df, natural_keys, source_sk_col_override, "SOURCE")
target_sk_col = detect_sk_col(target_df, natural_keys, target_sk_col_override, "TARGET")

print("Detected source SK column:", source_sk_col)
print("Detected target SK column:", target_sk_col)

source_h = add_duplicate_metadata(compute_hashes(source_df, natural_keys, is_scd2), "source", source_sk_col)
target_h = add_duplicate_metadata(compute_hashes(target_df, natural_keys, is_scd2), "target", target_sk_col)

version_grain = natural_keys + ([scd2_start_col] if is_scd2 else [])
print("Version grain:", version_grain)
print("Duplicate detection will be surfaced in map_status as DUPLICATE when a hash has multiple rows on either side.")


In [ ]:
src = source_h.alias("s")
tgt = target_h.alias("t")

join_cond = (F.col("s.match_hash") == F.col("t.match_hash")) & (F.col("s.source_row_num") == F.col("t.target_row_num"))

natural_key_expr = F.coalesce(F.col("s.natural_key"), F.col("t.natural_key"))
source_count_expr = F.coalesce(F.col("s.source_row_count"), F.lit(0))
target_count_expr = F.coalesce(F.col("t.target_row_count"), F.lit(0))

duplicate_expr = (source_count_expr > 1) | (target_count_expr > 1)

key_map_df = (
    src.join(tgt, join_cond, "full_outer")
    .select(
        natural_key_expr.alias("natural_key"),
        F.coalesce(F.col("s.match_hash"), F.col("t.match_hash")).alias("match_hash"),
        (
            F.coalesce(F.col(f"s.{scd2_start_col}"), F.col(f"t.{scd2_start_col}"))
            if is_scd2 else F.lit(None).cast("timestamp")
        ).alias("effectiveStartDate"),
        (
            F.coalesce(F.col(f"s.{scd2_end_col}"), F.col(f"t.{scd2_end_col}"))
            if is_scd2 else F.lit(None).cast("timestamp")
        ).alias("effectiveEndDate"),
        F.col(f"s.{source_sk_col}").cast("bigint").alias("old_sk"),
        F.col(f"t.{target_sk_col}").cast("bigint").alias("new_sk"),
        source_count_expr.alias("source_row_count"),
        target_count_expr.alias("target_row_count"),
        F.when(
            duplicate_expr,
            F.lit("DUPLICATE"),
        )
        .when(
            F.col(f"s.{source_sk_col}").isNotNull() & F.col(f"t.{target_sk_col}").isNotNull(),
            F.lit("MATCHED"),
        )
        .when(
            F.col(f"s.{source_sk_col}").isNotNull() & F.col(f"t.{target_sk_col}").isNull(),
            F.lit("ORPHAN_OLD"),
        )
        .otherwise(F.lit("ORPHAN_NEW"))
        .alias("map_status"),
    )
)

# cache() is not supported on Databricks serverless compute.
# Materialize to a Delta checkpoint instead so subsequent actions read the same data without recomputing.
_tmp_path = f"{temp_storage_path}/{source_table}_keymap_tmp"
key_map_df.write.mode("overwrite").format("delta").save(_tmp_path)
key_map_df = spark.read.format("delta").load(_tmp_path)

print("Key map status counts:")
key_map_df.groupBy("map_status").count().orderBy("map_status").show(truncate=False)

try:

    display(key_map_df)

except NameError:    key_map_df.show(50, truncate=False)

In [ ]:
if save_keymap:
    out_catalog = keymap_catalog if keymap_catalog else target_catalog
    out_fqn = f"{out_catalog}.{keymap_schema}.{keymap_table}"

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {out_catalog}.{keymap_schema}")
    (
        key_map_df
        .repartition(1)
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(out_fqn)
    )
    print(f"Saved key_map to {out_fqn}")
else:
    print("save_keymap=false, so key_map was not persisted")